# VQASynth++ Phase 5 — Stage 2: GRPO verifiable dense-reward RL

Starts from the **Stage-1 SFT adapter** (so outputs already parse into the 4-tag
format) and shapes grounded spatial reasoning with GRPO against the SpatialThinker
dense reward (`vqasynth.training.make_grpo_reward` → `spatial_reward.dense_reward`:
format + count + accuracy + CIoU, lexicographic gating).

**Why SFT-first is load-bearing:** the reward is format-gated (`I[format==1]`). An
un-cold-started model emits unparseable output → 0 reward → no gradient signal.
Stage 1 guarantees parseable `<observe>/<scene>/<think>/<answer>`; Stage 2 then
rewards *correct, box-grounded* reasoning.

Base = Qwen3-VL-8B-Instruct + the Stage-1 LoRA. GPU notebook — not run here.

> Note: VLM support in TRL's `GRPOTrainer` is version-sensitive — pin/align with
> what the harness uses. The reward itself is model-agnostic (scores decoded text
> + forwarded `grounded_state`/`answer` columns) so it is stable across versions.

## 0. Install

In [ ]:
# %pip install -q "trl>=0.12" "transformers>=4.57" accelerate peft datasets pillow qwen-vl-utils
# Optional (faster rollouts): %pip install -q vllm
# Restart the runtime after install (Colab).

## 1. Config

In [ ]:
BASE_ID = "Qwen/Qwen3-VL-8B-Instruct"     # match Stage 1 (arm #2: ...-Thinking)
SFT_ADAPTER = "qwen3vl8b_instruct_qv_vqasynthpp_sft"   # Stage-1 output dir
DATA_REPO = "remyxai/vqasynthpp_traces"    # same Stage-0 pipeline output
IMAGE_COL, MESSAGES_COL, ANSWER_COL, GROUNDED_COL = "image", "messages", "answer", "grounded_state"

OUTPUT_DIR = "qwen3vl8b_instruct_qv_vqasynthpp_grpo"
NUM_GENERATIONS = 8        # GRPO group size (completions sampled per prompt)
MAX_PROMPT_LEN, MAX_COMPLETION_LEN = 1024, 640
LR, BSZ, GRAD_ACCUM, EPOCHS = 1e-6, 1, 4, 1
USE_VLLM = False           # set True if vllm is installed (faster generation)
SEED = 0

## 2. Data — prompts + reward columns
GRPO generates the trace, so the dataset carries the **prompt** (image + question) plus the per-example `grounded_state` and `answer` the reward scores against. Reuse the Phase-4 gate so RL trains only on clean rows.

In [ ]:
from datasets import Dataset, load_dataset
from vqasynth import trace_validation as tv
from vqasynth import training as T

raw = list(load_dataset(DATA_REPO, split="train"))

def q_of(r):  return T.extract_question(r.get(MESSAGES_COL))
def gs_of(r): return r.get(GROUNDED_COL)
def an_of(r): return r.get(ANSWER_COL)

# gate (faithfulness on the GENERATED trace isn't checkable yet; grounding/dedup/
# calibration still apply to the source rows). Use trace_of=lambda: "" to skip
# trace-only checks and keep rows with a usable grounded_state + answer + question.
rows = [r for r in raw if q_of(r) and gs_of(r) and an_of(r) is not None]
calib = tv.chance_calibration([an_of(r) for r in rows])
assert not calib["degenerate"], f"degenerate answers: {calib}"
print(f"GRPO prompts: {len(rows)}  answer-dist: {calib['distribution']}")

def to_prompt(r):
    return [{"role": "user",
             "content": [{"type": "image", "image": r[IMAGE_COL]},
                         {"type": "text", "text": q_of(r)}]}]

ds = Dataset.from_list([
    {"prompt": to_prompt(r), "grounded_state": gs_of(r), "answer": an_of(r)}
    for r in rows
])

## 3. Reward — the dense CIoU reward adapter
`make_grpo_reward()` returns a `GRPOTrainer`-shaped reward: the trainer forwards the `grounded_state` / `answer` dataset columns as kwargs; it builds the GT scene per example and returns scalar totals. Smoke-test it on CPU before training.

In [ ]:
from vqasynth.training import make_grpo_reward, grpo_reward

reward_fn = make_grpo_reward()   # default SpatialThinker weights (0.1/0.2/0.5/0.2)

# CPU smoke: a correct, box-grounded completion vs a wrong one
_gs = ds[0]["grounded_state"]; _ans = ds[0]["answer"]
_good = ('<observe>x</observe><scene>{"objects":[{"id":0,"label":"obj","bbox":[0,0,10,10]}],'
         '"relations":[]}</scene><think>t</think><answer>' + str(_ans) + "</answer>")
print("breakdown (good):", grpo_reward([_good], [_gs], [_ans], return_breakdown=True)[0])
print("reward (wrong answer):", grpo_reward([_good.replace("answer>"+str(_ans), "answer>__nope__")], [_gs], [_ans])[0])

## 4. Model — base + Stage-1 LoRA (continue training the adapter)

In [ ]:
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import PeftModel

processor = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)
base = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_ID, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
)
# load the Stage-1 adapter and keep it trainable for RL
model = PeftModel.from_pretrained(base, SFT_ADAPTER, is_trainable=True)
model.config.use_cache = False
model.print_trainable_parameters()

## 5. GRPO train

In [ ]:
from trl import GRPOConfig, GRPOTrainer

args = GRPOConfig(
    output_dir=OUTPUT_DIR, learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BSZ, gradient_accumulation_steps=GRAD_ACCUM,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LEN, max_completion_length=MAX_COMPLETION_LEN,
    bf16=True, gradient_checkpointing=True, logging_steps=5, save_steps=100,
    use_vllm=USE_VLLM, seed=SEED, report_to=[], remove_unused_columns=False,
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],           # forwards grounded_state/answer columns
    args=args,
    train_dataset=ds,
    processing_class=processor,         # VLM: processor handles image+text prompts
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("saved GRPO student ->", OUTPUT_DIR)

## 6. Next — Phase 6 gating eval
Run the cheap gate first: **Qwen3-VL-8B base vs base + VQASynth++ (SFT→GRPO)**,
per-dimension (metric / multi-view / perspective / grounding), chance-normalized,
trace-scored. Contrast arms: **Thinking** base (swap `BASE_ID`/`SFT_ADAPTER`) and the
weak base (Qwen2.5-VL-3B). Expand to the full 3 harness arms only if the gate shows lift.

Watch the logged reward components during training: if `format`/`count` climb while
`accuracy`/`spatial` stay flat, that's reward-hacking — tighten the gate or weights.